In [1]:
# ==========================================
# CELL 1: INSTALASI LIBRARY
# ==========================================
%pip install -q xgboost lightgbm optuna rank_bm25 gensim Sastrawi tqdm pandas numpy scikit-learn scipy matplotlib seaborn

Note: you may need to restart the kernel to use updated packages.


In [2]:
# ==========================================
# CELL 2: IMPORT & KONFIGURASI
# ==========================================
import os, random, warnings
import numpy as np
import pandas as pd
from tqdm.auto import tqdm # Untuk menampilkan progress bar
tqdm.pandas()
warnings.filterwarnings('ignore')

# ML & NLP Libraries
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import scipy.sparse as sp

# Ensemble & Tuning
import lightgbm as lgb
import xgboost as xgb
import optuna

# Custom Text Features
from gensim.models import FastText
from rank_bm25 import BM25Okapi

# ---- Mengunci Seed untuk Reproduksibilitas ----
SEED = 42
def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

seed_everything(SEED)
print(f"✅ Seed locked globally at: {SEED}")

# Pastikan folder data tersedia
DATA_DIR = "./data"
print(f"📂 Direktori Data: {DATA_DIR}")

c:\Users\Ezra Faira\Documents\1. KULIAH\Lomba2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Seed locked globally at: 42
📂 Direktori Data: ./data


In [3]:
# ==========================================
# CELL 3: LOAD DATA & HANDLING MISSING/DUPLICATE
# ==========================================
# Asumsi file bernama train.csv dan test.csv di dalam folder /data
train_path = os.path.join(DATA_DIR, "train.csv")
test_path = os.path.join(DATA_DIR, "test.csv")

try:
    train = pd.read_csv(train_path)
    test = pd.read_csv(test_path)
    print("✅ Data berhasil dimuat!")
except FileNotFoundError:
    print("❌ File tidak ditemukan! Pastikan train.csv dan test.csv ada di folder /data")
    # Membuat dummy data agar kode tidak error saat kamu baca (HAPUS ini saat run betulan)
    train = pd.DataFrame({"title":["Banjir di Jakarta", "Kebakaran pasar"], "content":["Air meluap di bundaran HI", "Api hanguskan ruko"], "label":[1, 1]})
    test = pd.DataFrame({"title":["Gempa bumi"], "content":["Tanah bergetar"]})

# 1. Penanganan Duplikat (Hanya di Train)
print("\n=== CEK DUPLIKAT ===")
print("Duplikat awal di Train :", train.duplicated(subset=['title', 'content']).sum())
train = train.drop_duplicates(subset=['title', 'content'], keep='first').reset_index(drop=True)
print("Sisa duplikat di Train :", train.duplicated(subset=['title', 'content']).sum())

# 2. Penanganan Missing Values
print("\n=== CEK MISSING VALUES ===")
train = train.dropna(subset=["title", "content", "label"]).reset_index(drop=True)

test["title"] = test["title"].fillna("").astype(str)
test["content"] = test["content"].fillna("").astype(str)
print("Missing values dibersihkan.")

# Hitung rasio imbalanced untuk XGBoost (pos_scale_weight)
ratio = train['label'].value_counts()[0] / train['label'].value_counts()[1]
print(f"\nRasio Kelas (0/1) untuk XGBoost: {ratio:.2f}")

✅ Data berhasil dimuat!

=== CEK DUPLIKAT ===
Duplikat awal di Train : 3
Sisa duplikat di Train : 0

=== CEK MISSING VALUES ===
Missing values dibersihkan.

Rasio Kelas (0/1) untuk XGBoost: 0.11


In [4]:
# ==========================================
# CELL 4: PREPROCESSING (Pembersihan & Stemming)
# ==========================================
import re
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

stemmer = StemmerFactory().create_stemmer()
stopword = StopWordRemoverFactory().create_stop_word_remover()

def clean_text(text, max_words=None):
    if not isinstance(text, str): text = str(text)
    if max_words:
        text = " ".join(text.split()[:max_words])
        
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE) # Hapus URL
    text = re.sub(r'[^a-z\s]', ' ', text) # Hapus simbol & angka
    text = re.sub(r'\s+', ' ', text).strip() # Hapus spasi berlebih
    
    # Stemming & Stopword
    text = stopword.remove(text)
    text = stemmer.stem(text)
    return text

print("⏳ Memulai preprocessing data train... (Ini mungkin memakan waktu)")
train["clean_title"] = train["title"].progress_apply(lambda x: clean_text(x, max_words=None))
train["clean_content"] = train["content"].progress_apply(lambda x: clean_text(x, max_words=256))

print("⏳ Memulai preprocessing data test...")
test["clean_title"] = test["title"].progress_apply(lambda x: clean_text(x, max_words=None))
test["clean_content"] = test["content"].progress_apply(lambda x: clean_text(x, max_words=256))

⏳ Memulai preprocessing data train... (Ini mungkin memakan waktu)


100%|██████████| 14394/14394 [42:50<00:00,  5.60it/s] 


⏳ Memulai preprocessing data test...


100%|██████████| 3603/3603 [04:38<00:00, 12.94it/s]


In [ ]:
# ==========================================
# CELL 4.5: SIMPAN CHECKPOINT DATA BERSIH
# ==========================================
# Menyimpan hasil stemming ke dalam file CSV baru
# Index=False agar nomor baris tidak ikut tersimpan menjadi kolom baru
train.to_csv("train_cleaned.csv", index=False)
test.to_csv("test_cleaned.csv", index=False)

print("✅ Data hasil stemming berhasil disimpan!")

In [6]:
# ==========================================
# CELL 5: TF-IDF, BM25, & JACCARD
# ==========================================
print("⚙️ Ekstraksi Fitur TF-IDF & BM25...")

# 1. TF-IDF
all_text = train['clean_title'].tolist() + train['clean_content'].tolist() + \
           test['clean_title'].tolist() + test['clean_content'].tolist()

tfidf = TfidfVectorizer(ngram_range=(1, 2), max_features=10000)
tfidf.fit(all_text)

train_title_tfidf = tfidf.transform(train['clean_title'])
train_content_tfidf = tfidf.transform(train['clean_content'])
test_title_tfidf = tfidf.transform(test['clean_title'])
test_content_tfidf = tfidf.transform(test['clean_content'])

# 2. Setup BM25 Global (IDF dihitung dari seluruh konten berita)
all_contents_tokenized = [str(doc).split() for doc in (train['clean_content'].tolist() + test['clean_content'].tolist())]
bm25_model = BM25Okapi(all_contents_tokenized)

def compute_jaccard(title, content):
    set_t, set_c = set(str(title).split()), set(str(content).split())
    if not set_t or not set_c: return 0.0
    return len(set_t.intersection(set_c)) / len(set_t.union(set_c))

def compute_bm25_row(title, content, bm25_obj):
    """Menghitung skor BM25 spesifik antara 1 judul terhadap 1 isi berita (menggunakan IDF global)"""
    query = str(title).split()
    doc_words = str(content).split()
    doc_len = len(doc_words)
    score = 0.0
    for word in query:
        # PERBAIKAN: Menggunakan .idf (bukan .word_df)
        if word in bm25_obj.idf:
            idf = bm25_obj.idf[word]
            tf = doc_words.count(word)
            # Rumus standar BM25Okapi
            score += idf * (tf * (bm25_obj.k1 + 1)) / (tf + bm25_obj.k1 * (1 - bm25_obj.b + bm25_obj.b * (doc_len / bm25_obj.avgdl)))
    return score

for df, title_mat, content_mat in [(train, train_title_tfidf, train_content_tfidf), 
                                   (test, test_title_tfidf, test_content_tfidf)]:
    # Jaccard & Cosine TF-IDF
    df['jaccard_sim'] = df.apply(lambda x: compute_jaccard(x['clean_title'], x['clean_content']), axis=1)
    df['tfidf_cosine_sim'] = title_mat.multiply(content_mat).sum(axis=1).A1
    df['length_ratio'] = df['clean_content'].apply(lambda x: len(str(x).split())) / \
                         (df['clean_title'].apply(lambda x: len(str(x).split())) + 1e-5)
    
    # BM25 Score
    df['bm25_score'] = df.apply(lambda x: compute_bm25_row(x['clean_title'], x['clean_content'], bm25_model), axis=1)

print("✅ Fitur Lexical & BM25 selesai!")

⚙️ Ekstraksi Fitur TF-IDF & BM25...
✅ Fitur Lexical & BM25 selesai!


In [7]:
# ==========================================
# CELL 6: FASTTEXT SEMANTIC SIMILARITY
# ==========================================
print("🧠 Melatih FastText dari awal (Legal) ...")
sentences = [str(text).split() for text in all_text]

# Train FastText (menangkap makna kata & sub-word)
ft_model = FastText(sentences=sentences, vector_size=100, window=5, min_count=2, workers=4, seed=SEED)

def get_avg_ft_vector(text, model, vector_size=100):
    words = str(text).split()
    vectors = [model.wv[w] for w in words if w in model.wv]
    if not vectors: return np.zeros(vector_size)
    return np.mean(vectors, axis=0)

for df in [train, test]:
    title_ft = np.array([get_avg_ft_vector(t, ft_model) for t in df['clean_title']])
    content_ft = np.array([get_avg_ft_vector(c, ft_model) for c in df['clean_content']])
    
    # Cosine Similarity Teroptimasi Array
    dot_product = np.sum(title_ft * content_ft, axis=1)
    norm_title = np.linalg.norm(title_ft, axis=1)
    norm_content = np.linalg.norm(content_ft, axis=1)
    
    df['fasttext_cosine_sim'] = dot_product / (norm_title * norm_content + 1e-10)

print("✅ Fitur Semantik FastText selesai!")

🧠 Melatih FastText dari awal (Legal) ...
✅ Fitur Semantik FastText selesai!


In [8]:
# ==========================================
# CELL 7: GABUNGAN SELURUH FITUR MATRIKS
# ==========================================
print("🔗 Menggabungkan matriks (TF-IDF + Features)...")

# Daftar kolom fitur numerik (Perhatikan penambahan bm25_score dan perubahan fasttext)
extra_cols = ['jaccard_sim', 'tfidf_cosine_sim', 'length_ratio', 'bm25_score', 'fasttext_cosine_sim']

X_train = sp.hstack([train_title_tfidf, train_content_tfidf, train[extra_cols].values]).tocsr()
X_test = sp.hstack([test_title_tfidf, test_content_tfidf, test[extra_cols].values]).tocsr()
y_train = train['label'].values

print(f"Dimensi X_train: {X_train.shape}")

🔗 Menggabungkan matriks (TF-IDF + Features)...
Dimensi X_train: (14394, 20005)


In [ ]:
# ==========================================
# CELL 8: OPTUNA TUNING (LIGHTGBM & XGBOOST)
# ==========================================
# Set jumlah trial kecil (misal 10) untuk testing. Saat training final, naikkan ke 30-50.
N_TRIALS = 50 

def objective_lgb(trial):
    params = {
        'objective': 'binary',
        'metric': 'binary_logloss',
        'class_weight': 'balanced',
        'boosting_type': 'gbdt',
        'random_state': SEED,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1),
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'num_leaves': trial.suggest_int('num_leaves', 20, 100),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'verbose': -1
    }
    
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
    f1_scores = []
    
    for train_idx, val_idx in skf.split(X_train, y_train):
        X_tr, y_tr = X_train[train_idx], y_train[train_idx]
        X_val, y_val = X_train[val_idx], y_train[val_idx]
        
        model = lgb.LGBMClassifier(**params)
        model.fit(X_tr, y_tr)
        preds = (model.predict_proba(X_val)[:, 1] >= 0.5).astype(int)
        f1_scores.append(f1_score(y_val, preds, average='macro'))
        
    return np.mean(f1_scores)

print("🔍 Memulai Optuna Tuning untuk LightGBM...")
optuna.logging.set_verbosity(optuna.logging.WARNING) # Sembunyikan log agar rapi
study_lgb = optuna.create_study(direction='maximize')
study_lgb.optimize(objective_lgb, n_trials=N_TRIALS)

best_lgb_params = study_lgb.best_params
best_lgb_params['class_weight'] = 'balanced'
best_lgb_params['random_state'] = SEED
best_lgb_params['verbose'] = -1
print("✅ Best LightGBM Params:", best_lgb_params)

🔍 Memulai Optuna Tuning untuk LightGBM...


In [ ]:
# ==========================================
# CELL 9: K-FOLD CROSS VALIDATION ENSEMBLE
# ==========================================
n_folds = 5
skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=SEED)

oof_preds_lgb = np.zeros(X_train.shape[0])
test_preds_lgb = np.zeros(X_test.shape[0])

oof_preds_xgb = np.zeros(X_train.shape[0])
test_preds_xgb = np.zeros(X_test.shape[0])

# Parameter basic XGBoost dengan scale_pos_weight untuk imbalance
xgb_params = {
    'random_state': SEED,
    'learning_rate': 0.05,
    'n_estimators': 300,
    'max_depth': 6,
    'scale_pos_weight': ratio, # dari perhitungan EDA di awal
    'eval_metric': 'logloss',
    'use_label_encoder': False
}

print("⚙️ Melatih Model Ensemble (LightGBM + XGBoost)...")
for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    X_tr, y_tr = X_train[train_idx], y_train[train_idx]
    X_val, y_val = X_train[val_idx], y_train[val_idx]
    
    # 1. LIGHTGBM
    model_lgb = lgb.LGBMClassifier(**best_lgb_params)
    # Menggunakan callback early stopping yang direkomendasikan LightGBM terbaru
    model_lgb.fit(X_tr, y_tr,
                  eval_set=[(X_val, y_val)],
                  callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=False)])
    
    oof_preds_lgb[val_idx] = model_lgb.predict_proba(X_val)[:, 1]
    test_preds_lgb += model_lgb.predict_proba(X_test)[:, 1] / n_folds
    
    # 2. XGBOOST
    model_xgb = xgb.XGBClassifier(**xgb_params)
    model_xgb.fit(X_tr, y_tr,
                  eval_set=[(X_val, y_val)],
                  verbose=False)
    
    oof_preds_xgb[val_idx] = model_xgb.predict_proba(X_val)[:, 1]
    test_preds_xgb += model_xgb.predict_proba(X_test)[:, 1] / n_folds
    
    print(f"Fold {fold+1} Selesai.")

# ENSEMBLE PROBABILITAS (Rata-rata 50:50)
oof_preds_ensemble = (oof_preds_lgb * 0.5) + (oof_preds_xgb * 0.5)
test_preds_ensemble = (test_preds_lgb * 0.5) + (test_preds_xgb * 0.5)

In [ ]:
# ==========================================
# CELL 10: THRESHOLD TUNING (MACRO F1)
# ==========================================
print("\n=== MENCARI THRESHOLD OPTIMAL (MACRO F1) ===")
best_th, best_f1 = 0.5, 0.0

for th in np.arange(0.1, 0.9, 0.01):
    f1 = f1_score(y_train, (oof_preds_ensemble >= th).astype(int), average="macro")
    if f1 > best_f1:
        best_f1, best_th = f1, th

print(f"Macro F1 @0.50             : {f1_score(y_train, (oof_preds_ensemble>=0.5).astype(int), average='macro'):.4f}")
print(f"Threshold optimal          : {best_th:.2f}")
print(f"Macro F1 @threshold opt    : {best_f1:.4f}")

# Dapatkan prediksi OOF final berdasar threshold
final_oof_classes = (oof_preds_ensemble >= best_th).astype(int)
print("\nClassification report (OOF, threshold optimal):")
print(classification_report(y_train, final_oof_classes, target_names=["Tidak Sesuai (0)", "Sesuai (1)"]))

In [ ]:
# ==========================================
# CELL 11: EXPORT SUBMISSION (Penamaan File Otomatis)
# ==========================================
import os

# Menentukan kelas test menggunakan threshold optimal
final_test_preds = (test_preds_ensemble >= best_th).astype(int)

submission = pd.DataFrame({
    'id': test.index, # Sesuaikan jika kolom ID di dataset aslimu bernama lain
    'label': final_test_preds
})

# Logika Penamaan File Auto-Increment (submission, submission-e, submission-e2, dst)
base_name = "submission"
ext = ".csv"
file_name = f"{base_name}{ext}"

# Cek apakah submission.csv sudah ada
if os.path.exists(file_name):
    # Cek untuk submission-e.csv
    file_name = f"{base_name}-e{ext}"
    counter = 2
    
    # Cek terus untuk submission-e2.csv, submission-e3.csv, dst
    while os.path.exists(file_name):
        file_name = f"{base_name}-e{counter}{ext}"
        counter += 1

# Export DataFrame ke CSV dengan nama yang sudah divalidasi
submission.to_csv(file_name, index=False)
print(f"🏆 File {file_name} berhasil dibuat dan siap di-submit!")

In [ ]:
# ==========================================
# CELL 12: VISUALISASI DATA & MODEL
# ==========================================
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import seaborn as sns

os.makedirs("visualisasi", exist_ok=True)

# 1. DISTRIBUSI KELAS
def plot_distribusi_kelas(train):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    counts = train['label'].value_counts().sort_index()
    labels = ['Tidak Sesuai (0)', 'Sesuai (1)']
    colors = ['#e74c3c', '#2ecc71']
    
    axes[0].bar(labels, counts.values, color=colors)
    axes[0].set_title('Distribusi Kelas (Bar Chart)', fontsize=14)
    axes[0].set_ylabel('Jumlah Sampel')
    for i, v in enumerate(counts.values):
        axes[0].text(i, v + (max(counts)*0.02), str(v), ha='center', fontweight='bold')

    axes[1].pie(counts.values, labels=labels, autopct='%1.1f%%', colors=colors, startangle=90, explode=(0.05, 0))
    axes[1].set_title('Distribusi Kelas (Pie Chart)', fontsize=14)
    
    plt.tight_layout()
    plt.savefig('visualisasi/01_distribusi_kelas.png', dpi=150, bbox_inches='tight')
    plt.close()

# 2. DISTRIBUSI PANJANG
def plot_distribusi_panjang(train):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    title_len = train['title'].astype(str).apply(lambda x: len(x.split()))
    content_len = train['content'].astype(str).apply(lambda x: len(x.split()))
    
    axes[0].hist(title_len, bins=50, color='#3498db', edgecolor='black', alpha=0.7)
    axes[0].set_title('Distribusi Panjang Judul', fontsize=14)
    axes[0].axvline(title_len.median(), color='red', linestyle='--', label=f'Median: {title_len.median():.0f}')
    axes[0].legend()

    axes[1].hist(content_len, bins=50, color='#e67e22', edgecolor='black', alpha=0.7)
    axes[1].set_title('Distribusi Panjang Isi Berita', fontsize=14)
    axes[1].axvline(content_len.median(), color='red', linestyle='--', label=f'Median: {content_len.median():.0f}')
    axes[1].legend()

    plt.tight_layout()
    plt.savefig('visualisasi/02_distribusi_panjang_kata.png', dpi=150, bbox_inches='tight')
    plt.close()

# 3. DIAGRAM ALUR (Disesuaikan untuk Ensemble Pipeline)
def plot_arsitektur_pipeline():
    fig, ax = plt.subplots(figsize=(14, 8))
    ax.set_xlim(0, 10); ax.set_ylim(0, 10); ax.axis('off')

    # Paths
    wide_boxes = [(1.5, 8.5, 'Teks\nInput'), (3.5, 8.5, 'TF-IDF\n(N-Gram)'), (6.5, 8.5, 'LightGBM\nModel')]
    deep_boxes = [(1.5, 5, 'Teks\nInput'), (3.5, 5, 'FastText\n(Sub-word)'), (6.5, 5, 'XGBoost\nModel')]
    feat_box = (4.0, 2.5, 'Feature Engineering\n(Jaccard, BM25, FastText Sim)')
    merge_box = (8.5, 6.75, 'Ensemble\n(Rata-rata Prob)')
    output_box = (10, 6.75, 'Output\n(Threshold)')

    # Draw boxes
    for x, y, text in wide_boxes:
        ax.add_patch(mpatches.FancyBboxPatch((x-0.7, y-0.4), 1.4, 0.8, boxstyle="round,pad=0.1", facecolor='#3498db', edgecolor='black', linewidth=2))
        ax.text(x, y, text, ha='center', va='center', fontsize=9, fontweight='bold', color='white')
    
    for x, y, text in deep_boxes:
        ax.add_patch(mpatches.FancyBboxPatch((x-0.7, y-0.4), 1.4, 0.8, boxstyle="round,pad=0.1", facecolor='#2ecc71', edgecolor='black', linewidth=2))
        ax.text(x, y, text, ha='center', va='center', fontsize=9, fontweight='bold', color='white')
        
    ax.add_patch(mpatches.FancyBboxPatch((feat_box[0]-1.4, feat_box[1]-0.4), 2.8, 0.8, boxstyle="round,pad=0.1", facecolor='#f39c12', edgecolor='black', linewidth=2))
    ax.text(feat_box[0], feat_box[1], feat_box[2], ha='center', va='center', fontsize=9, fontweight='bold', color='white')

    ax.add_patch(mpatches.FancyBboxPatch((merge_box[0]-0.7, merge_box[1]-0.4), 1.4, 0.8, boxstyle="round,pad=0.1", facecolor='#e74c3c', edgecolor='black', linewidth=2))
    ax.text(merge_box[0], merge_box[1], merge_box[2], ha='center', va='center', fontsize=9, fontweight='bold', color='white')
    
    ax.add_patch(mpatches.FancyBboxPatch((output_box[0]-0.6, output_box[1]-0.4), 1.2, 0.8, boxstyle="round,pad=0.1", facecolor='#9b59b6', edgecolor='black', linewidth=2))
    ax.text(output_box[0], output_box[1], output_box[2], ha='center', va='center', fontsize=9, fontweight='bold', color='white')

    # Draw arrows (simplified)
    ax.annotate('', xy=(3.5-0.7, 8.5), xytext=(1.5+0.7, 8.5), arrowprops=dict(arrowstyle='->', lw=2, color='#3498db'))
    ax.annotate('', xy=(6.5-0.7, 8.5), xytext=(3.5+0.7, 8.5), arrowprops=dict(arrowstyle='->', lw=2, color='#3498db'))
    ax.annotate('', xy=(3.5-0.7, 5), xytext=(1.5+0.7, 5), arrowprops=dict(arrowstyle='->', lw=2, color='#2ecc71'))
    ax.annotate('', xy=(6.5-0.7, 5), xytext=(3.5+0.7, 5), arrowprops=dict(arrowstyle='->', lw=2, color='#2ecc71'))
    ax.annotate('', xy=(merge_box[0]-0.7, merge_box[1]+0.2), xytext=(6.5+0.7, 8.5), arrowprops=dict(arrowstyle='->', lw=2, color='#e74c3c'))
    ax.annotate('', xy=(merge_box[0]-0.7, merge_box[1]-0.2), xytext=(6.5+0.7, 5), arrowprops=dict(arrowstyle='->', lw=2, color='#e74c3c'))
    ax.annotate('', xy=(output_box[0]-0.6, output_box[1]), xytext=(merge_box[0]+0.7, merge_box[1]), arrowprops=dict(arrowstyle='->', lw=2, color='black'))
    
    ax.set_title('Arsitektur Ensemble ML Klasik (Sesuai Rules)', fontsize=16, fontweight='bold', pad=20)
    plt.savefig('visualisasi/03_arsitektur_pipeline.png', dpi=150, bbox_inches='tight')
    plt.close()

# 4. CONFUSION MATRIX
def plot_confusion_matrix(y_true, y_pred):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    cm = confusion_matrix(y_true, y_pred)
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0], xticklabels=['Tidak Sesuai', 'Sesuai'], yticklabels=['Tidak Sesuai', 'Sesuai'])
    axes[0].set_title('Confusion Matrix (Absolut)', fontsize=14)
    
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Blues', ax=axes[1], xticklabels=['Tidak Sesuai', 'Sesuai'], yticklabels=['Tidak Sesuai', 'Sesuai'])
    axes[1].set_title('Confusion Matrix (Ternormalisasi)', fontsize=14)
    
    plt.tight_layout()
    plt.savefig('visualisasi/04_confusion_matrix.png', dpi=150, bbox_inches='tight')
    plt.close()

# 5. DISTRIBUSI FITUR
def plot_distribusi_fitur(train):
    # Fitur disesuaikan dengan FastText dan BM25
    features = ['jaccard_sim', 'bm25_score', 'fasttext_cosine_sim']
    titles = ['Jaccard Similarity', 'BM25 Score', 'FastText Cosine Similarity']
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    for ax, feat, title in zip(axes, features, titles):
        if feat not in train.columns: continue
        for label, color, name in [(0, '#e74c3c', 'Tidak Sesuai'), (1, '#2ecc71', 'Sesuai')]:
            subset = train[train['label'] == label][feat]
            ax.hist(subset, bins=50, alpha=0.6, color=color, label=name, density=True)
        ax.set_title(f'Distribusi {title}', fontsize=12)
        ax.legend()

    plt.tight_layout()
    plt.savefig('visualisasi/05_distribusi_fitur.png', dpi=150, bbox_inches='tight')
    plt.close()

print("\n⚙️ GENERATE VISUALISASI...")
plot_distribusi_kelas(train)
plot_distribusi_panjang(train)
plot_arsitektur_pipeline()
plot_confusion_matrix(y_train, final_oof_classes)
plot_distribusi_fitur(train)
print("✅ Semua file visualisasi sukses disimpan di folder /visualisasi")